# Tutorial 05 — Generate Your Own Images

**What you will learn:**
- How to use `MAISI_RT_Generate.py` to generate any of the 16 presets
- What a YAML config file contains and how to read it
- How seeds control reproducibility
- How to explore all available anatomy structures
- How to prepare and run your first GPU generation

**This notebook has two parts:**
1. Everything before the GPU run — you can do this without a GPU, exploring configs and data
2. The actual generation step — requires a GPU and NVIDIA model weights

---
> **Safety reminder:** All images are fully synthetic — generated by AI, not from any real patient.

In [ ]:
from pathlib import Path
import subprocess, sys, json, yaml
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path().resolve()
for _ in range(6):
    if (PROJECT_ROOT / 'MAISI_RT_Generate.py').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PYTHON = sys.executable
GENERATE_SCRIPT = PROJECT_ROOT / 'MAISI_RT_Generate.py'

%matplotlib inline
plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
                     'text.color': 'white', 'axes.labelcolor': '#aaa',
                     'xtick.color': '#aaa', 'ytick.color': '#aaa'})

print('Project root:', PROJECT_ROOT)
print('Generate script found:', GENERATE_SCRIPT.exists())

## Part 1 — Exploring the generation script (no GPU)

The entire generation interface lives in a single script: `MAISI_RT_Generate.py`.

You can use it in three ways:

| Command | What happens |
|---|---|
| `python MAISI_RT_Generate.py` | Shows an interactive menu |
| `python MAISI_RT_Generate.py --list` | Lists all 16 presets with their names |
| `python MAISI_RT_Generate.py --preset <name> --dry-run` | Writes the config, but does NOT use the GPU |

Let's start by listing all available presets.

In [ ]:
# ── List all available presets ─────────────────────────────────────────────
result = subprocess.run(
    [PYTHON, str(GENERATE_SCRIPT), '--list'],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)

## Choosing a preset

Each preset name is what you pass to `--preset`. For example:

- `ct_lungs_tumor` → CT chest with all lung lobes and a lung tumor mask
- `brain_t1` → T1-weighted whole brain MRI
- `ct_pelvis_rt` → Pelvic CT with prostate, bladder, femur, and hip bones

The preset controls:
- Which AI model runs (CT paired, brain MRI, or body MRI)
- Which structures appear in the mask
- The image resolution and field of view

Let's do a **dry run** — this writes the config file without using the GPU.

In [ ]:
# ── Dry run — write the config without running the GPU ─────────────────────
#
# Change PRESET_NAME to whichever preset interests you most.

PRESET_NAME = 'ct_lungs_tumor'   # ← change this to any preset from --list
SEED = 55001                      # ← any integer, controls which anatomy is generated
GPU  = 1                          # ← GPU index (0 or 1 depending on your machine)

result = subprocess.run(
    [PYTHON, str(GENERATE_SCRIPT),
     '--preset', PRESET_NAME,
     '--seed',   str(SEED),
     '--gpu',    str(GPU),
     '--dry-run'],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# ── Read and explain the generated YAML config ────────────────────────────

yaml_path = PROJECT_ROOT / f'configs/maisi2/quickgen_{PRESET_NAME}.yaml'

if yaml_path.exists():
    with open(yaml_path) as f:
        config = yaml.safe_load(f)
    print(f'Config file: {yaml_path.relative_to(PROJECT_ROOT)}\n')
    print('─' * 60)
    print(yaml_path.read_text())
else:
    print(f'Config not found at {yaml_path} — re-run the dry-run cell above.')

## Understanding the config file

The YAML config controls everything about a generation run. Here are the most important fields:

### For CT with masks (`ct_structures` block)

| Field | What it does | Safe to change? |
|---|---|---|
| `seeds` | Which seeds to generate. Add more integers for more cases | Yes |
| `anatomy_list` | Which structures appear in the mask | Yes — see `docs/config_value_reference.md` |
| `body_region` | Coarse region hint | Usually leave as-is |
| `output_size` | Volume dimensions in voxels | Change carefully — must match anatomy size |
| `spacing` | mm per voxel | Change carefully — affects field of view |
| `num_inference_steps` | 30 = fast, quality trade-off | Leave at 30 for speed |
| `gpu_index` | Which GPU to use | Change to 0 if you only have one GPU |

### The field of view rule

```
FOV in mm = output_size × spacing
```

For the default CT settings: `256 × 1.5 = 384 mm` in X/Y and `128 × 2.0 = 256 mm` in Z. That is a normal chest-sized volume. If you change one, adjust the other to keep the FOV anatomically realistic.

In [ ]:
# ── Explore all available anatomy structure names ──────────────────────────
#
# MAISI supports 132 structures. Here we load the full list and display it
# grouped by body region.

label_dict_path = PROJECT_ROOT / 'external/NV-Generate-CTMR/configs/label_dict.json'

if label_dict_path.exists():
    label_dict = json.loads(label_dict_path.read_text())
    # Filter out dummy entries
    real_labels = {name: label_id for name, label_id in label_dict.items()
                   if 'dummy' not in name.lower()}

    print(f'Total real structures in MAISI: {len(real_labels)}\n')
    print(f'{"Structure name":<40} {"Label ID":>10}')
    print('─' * 54)
    for name, lid in sorted(real_labels.items(), key=lambda x: x[1]):
        print(f'{name:<40} {lid:>10}')
else:
    print('label_dict.json not found.')
    print(f'Expected at: {label_dict_path}')
    print('Clone the NVIDIA repo first: see docs/nvidia_setup_notes.md')

In [ ]:
# ── How to add structures to a preset ─────────────────────────────────────
#
# Want the ct_lungs_tumor preset but also include the aorta and trachea?
# 1. Run --dry-run to generate the YAML
# 2. Open the YAML and edit anatomy_list
# 3. Run the script directly
#
# Here we demonstrate this programmatically:

if yaml_path.exists():
    with open(yaml_path) as f:
        config = yaml.safe_load(f)

    block = config.get('ct_structures') or config.get('nvidia', {})
    current_anatomy = block.get('anatomy_list', [])

    print('Current anatomy_list:')
    for a in current_anatomy:
        print(f'  - {a}')
    print()

    # Suggest valid additions
    if label_dict_path.exists():
        suggestions = ['aorta', 'esophagus', 'spinal cord', 'thyroid gland']
        new_structures = [s for s in suggestions if s not in current_anatomy]
        print('You could add any of these to anatomy_list:')
        for s in new_structures:
            print(f'  + {s}')
        print()
        print('Edit the YAML file, then run:')
        print(f'  python scripts/run_nvidia_ct_structures_from_config.py \\')
        print(f'         --config {yaml_path.relative_to(PROJECT_ROOT)}')

## Understanding Seeds

A **seed** is an integer that initialises the random number generator inside the AI model. Think of it as which "alternative universe" anatomy to generate.

- **Same preset + same seed** → identical image every time (reproducible)
- **Same preset + different seed** → different anatomy, same structures

This is how you build a synthetic dataset: run the same preset with seeds 10001, 10002, 10003, … to get diverse but consistently labelled cases.

In [ ]:
# ── Seed diversity — comparing existing brain MRI seeds ───────────────────
#
# The brain MRI dataset was generated with seed 20260522.
# If you generate again with seed 99999, you'll get a different brain anatomy.
# Here we compare all existing T1 files if multiple seeds exist.

brain_dir = PROJECT_ROOT / 'outputs/maisi2_mr_brain_all_contrasts'
t1_files = sorted([f for f in brain_dir.glob('*mri_t1_seed*.nii.gz')
                   if 'skull_stripped' not in f.name])

print(f'Available T1 files: {len(t1_files)}')
for f in t1_files:
    print(f'  {f.name}')

if len(t1_files) >= 2:
    fig, axes = plt.subplots(1, len(t1_files), figsize=(6 * len(t1_files), 6))
    axes = axes if len(t1_files) > 1 else [axes]
    for ax, fp in zip(axes, t1_files):
        vol = nib.load(str(fp)).get_fdata(dtype=np.float32)
        z = vol.shape[2] // 2
        slc = vol[:, :, z]
        fg = slc[slc > slc.mean() * 0.1]
        lo, hi = np.percentile(fg, 1), np.percentile(fg, 99)
        ax.imshow(np.rot90(np.clip((slc - lo) / (hi - lo + 1e-6), 0, 1)), cmap='gray', aspect='equal')
        seed = fp.stem.split('seed')[1].split('_')[0]
        ax.set_title(f'Seed {seed}', fontsize=11)
        ax.axis('off')
    fig.suptitle('Same contrast (T1), different seeds → different brain anatomy', fontsize=12)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'figures/tut05_seed_comparison.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
else:
    print('Only one T1 file found. To see seed diversity, generate another seed:')
    print('  python MAISI_RT_Generate.py --preset brain_t1 --seed 99999')
    print('Then re-run this cell.')

## Part 2 — Running an actual generation (requires GPU)

Before running the next cell, make sure:

1. **You have a CUDA-capable GPU** (at least a 16GB VRAM)
2. **NVIDIA model weights are downloaded** into `external/NV-Generate-CTMR/`
   - See `docs/nvidia_setup_notes.md` for download instructions
3. **You are running in the MAISI_venv** environment

If you are not sure, run `--dry-run` first (done above) and inspect the config.

---

**Expected time:** 3–8 minutes per sample on a modern GPU.

In [ ]:
# ── ACTUAL GENERATION — runs on GPU ───────────────────────────────────────
#
# Remove the 'dry_run = True' line below when you are ready to generate.
# This cell runs MAISI_RT_Generate.py and streams output in real time.

dry_run = True   # ← SET TO False WHEN YOU HAVE A GPU AND MODEL WEIGHTS

cmd = [PYTHON, str(GENERATE_SCRIPT),
       '--preset', PRESET_NAME,
       '--seed',   str(SEED),
       '--gpu',    str(GPU)]

if dry_run:
    cmd.append('--dry-run')
    print('Running in dry-run mode. Set dry_run = False to use the GPU.')

print('Command:', ' '.join(str(c) for c in cmd))
print()

# Run and stream output line by line
with subprocess.Popen(cmd, cwd=PROJECT_ROOT,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True) as proc:
    for line in proc.stdout:
        print(line, end='')

if proc.returncode == 0:
    print('\nGeneration complete (or dry-run finished).')
else:
    print(f'\nProcess exited with code {proc.returncode}. Check the output above.')

In [ ]:
# ── Inspect the generated output ──────────────────────────────────────────
#
# After a successful (non-dry-run) generation, this cell shows your results.

output_dir = PROJECT_ROOT / f'outputs/quickgen_{PRESET_NAME}'

if not output_dir.exists():
    print(f'Output directory not found: {output_dir}')
    print('Run the generation cell above with dry_run = False first.')
else:
    niftis = sorted(output_dir.rglob('*.nii.gz'))
    pngs   = sorted(output_dir.rglob('*.png'))
    print(f'Output directory: {output_dir.relative_to(PROJECT_ROOT)}')
    print(f'  NIfTI files : {len(niftis)}')
    print(f'  PNG files   : {len(pngs)}')
    print()

    for f in niftis:
        print(f'  {f.relative_to(output_dir)}')

    # Display the orthogonal panel if it exists
    panels = list(output_dir.rglob('*orthogonal_panel.png'))
    if panels:
        print(f'\nDisplaying: {panels[0].name}')
        panel_img = plt.imread(str(panels[0]))
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.imshow(panel_img)
        ax.axis('off')
        ax.set_title(f'Generated: {PRESET_NAME}  seed={SEED}', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print('No orthogonal panel found — generation may not have completed.')

## Building a small dataset

To generate multiple cases in one command, edit the YAML config to include multiple seeds:

```yaml
ct_structures:
  seeds: [10001, 10002, 10003, 10004, 10005]
```

Then run:

```bash
python scripts/run_nvidia_ct_structures_from_config.py \
       --config configs/maisi2/quickgen_ct_lungs_tumor.yaml
```

Each seed produces an independent `sample_00N_seedNNN/` folder with its own NIfTI pair and PNG galleries.

In [ ]:
# ── Write a multi-seed config ──────────────────────────────────────────────
#
# This demonstrates how to modify the config for dataset generation.
# It saves the modified config but does NOT run generation.

DATASET_SEEDS = [10001, 10002, 10003]

if yaml_path.exists():
    with open(yaml_path) as f:
        config = yaml.safe_load(f)

    # Modify the seeds list
    if 'ct_structures' in config:
        config['ct_structures']['seeds'] = DATASET_SEEDS
    elif 'nvidia' in config:
        config['nvidia']['seeds'] = DATASET_SEEDS

    multi_yaml = PROJECT_ROOT / f'configs/maisi2/dataset_{PRESET_NAME}.yaml'
    with open(multi_yaml, 'w') as f:
        f.write(f'# Multi-seed dataset config — {PRESET_NAME}\n')
        f.write(f'# Seeds: {DATASET_SEEDS}\n\n')
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)

    print(f'Dataset config written to: {multi_yaml.relative_to(PROJECT_ROOT)}')
    print(f'Seeds: {DATASET_SEEDS}')
    print()
    print('To generate all cases (requires GPU):')
    if 'ct_structures' in config:
        print(f'  python scripts/run_nvidia_ct_structures_from_config.py \\')
    else:
        print(f'  python scripts/run_nvidia_case_from_config.py \\')
    print(f'         --config {multi_yaml.relative_to(PROJECT_ROOT)}')
else:
    print('Run the dry-run cell first to generate the base config.')

## Summary

| Step | Command | Notes |
|---|---|---|
| See all presets | `python MAISI_RT_Generate.py --list` | |
| Explore config | `python MAISI_RT_Generate.py --preset NAME --dry-run` | No GPU needed |
| Generate one case | `python MAISI_RT_Generate.py --preset NAME --seed N` | GPU required |
| Generate a dataset | Edit seeds in the YAML, run the script directly | GPU required |
| Add custom structures | Edit `anatomy_list` in the YAML | See `docs/config_value_reference.md` |

### Complete anatomy structure reference

All 132 available MAISI structure names are in `docs/config_value_reference.md`.
You can add any of them to `anatomy_list` to include them in the generated mask.

### Where your files go

```
outputs/quickgen_<preset>/
  visuals/
    sample_001_seed<N>/
      ct_seed<N>_image.nii.gz     ← CT volume
      ct_seed<N>_label.nii.gz     ← structure masks
      ct_orthogonal_panel.png     ← instant visual QA
      ct_structure_overlay_*.png  ← labelled overlays
```

These NIfTI files can be opened directly in **3D Slicer** (free at slicer.org) for full 3D exploration.

---

You have now completed all five tutorials. You know how CT and MRI data is structured, what windowing means, how segmentation masks work, what MRI contrasts are, and how to generate your own synthetic datasets. Happy generating!